In [33]:
from abc import ABC, abstractmethod
from datetime import datetime
import uuid


# ─── Base Payment Class ───────────────────────────────────────────────────────
class Payment(ABC):
    """Abstract base class for all payment types."""

    def __init__(self, amount, currency="INR"):
        if amount <= 0:
            raise ValueError("Amount must be positive!")
        self.__amount = amount
        self.currency = currency
        self.transaction_id = str(uuid.uuid4())[:8].upper()
        self.timestamp = datetime.now()
        self.status = "PENDING"

    @property
    def amount(self):
        return self.__amount

    @abstractmethod
    def process(self):
        """Each payment type must implement this."""
        pass

    @abstractmethod
    def refund(self):
        """Each payment type must implement refund."""
        pass

    def get_receipt(self):
        return (
            f"─── Receipt ───────────────────\n"
            f"  TXN ID   : {self.transaction_id}\n"
            f"  Amount   : {self.currency} {self.amount:,.2f}\n"
            f"  Status   : {self.status}\n"
            f"  Time     : {self.timestamp.strftime('%Y-%m-%d %H:%M:%S')}\n"
            f"  Type     : {self.__class__.__name__}\n"
            f"───────────────────────────────"
        )

    def __str__(self):
        return f"{self.__class__.__name__}(id={self.transaction_id}, amount={self.amount}, status={self.status})"

    def __repr__(self):
        return f"{self.__class__.__name__}(amount={self.amount!r}, currency={self.currency!r})"


# ─── Credit Card Payment ─────────────────────────────────────────────────────
class CreditCardPayment(Payment):
    def __init__(self, amount, card_number, card_holder):
        super().__init__(amount)
        self.__card_number = card_number   # Private
        self.card_holder = card_holder
        self.__masked_card = f"****-****-****-{card_number[-4:]}"

    def process(self):
        # Simulate validation + processing
        if len(self.__card_number) != 16:
            self.status = "FAILED"
            return f"Payment FAILED: Invalid card number."
        print(f"Processing card {self.__masked_card}...")
        self.status = "SUCCESS"
        return f"✅ Card payment of {self.currency} {self.amount:,.2f} successful!"

    def refund(self):
        if self.status != "SUCCESS":
            return "Cannot refund — payment not successful."
        self.status = "REFUNDED"
        return f"♻️  Refund of {self.currency} {self.amount:,.2f} to {self.__masked_card} initiated."

    def __str__(self):
        return f"CreditCardPayment({self.__masked_card}, {self.currency} {self.amount:,.2f})"


# ─── UPI Payment ─────────────────────────────────────────────────────────────
class UPIPayment(Payment):
    def __init__(self, amount, upi_id):
        super().__init__(amount)
        self.upi_id = upi_id

    def process(self):
        if "@" not in self.upi_id:
            self.status = "FAILED"
            return "Payment FAILED: Invalid UPI ID."
        print(f"Sending payment request to {self.upi_id}...")
        self.status = "SUCCESS"
        return f"✅ UPI payment of {self.currency} {self.amount:,.2f} to {self.upi_id} successful!"

    def refund(self):
        if self.status != "SUCCESS":
            return "Cannot refund — payment not successful."
        self.status = "REFUNDED"
        return f"♻️  Refund of {self.currency} {self.amount:,.2f} to {self.upi_id} initiated."


# ─── Wallet Payment ───────────────────────────────────────────────────────────
class WalletPayment(Payment):
    def __init__(self, amount, wallet_balance):
        super().__init__(amount)
        self.__wallet_balance = wallet_balance

    def process(self):
        if self.__wallet_balance < self.amount:
            self.status = "FAILED"
            return f"Payment FAILED: Insufficient wallet balance ({self.__wallet_balance} available)."
        self.__wallet_balance -= self.amount
        self.status = "SUCCESS"
        return f"✅ Wallet payment of {self.currency} {self.amount:,.2f} successful! Remaining: {self.__wallet_balance:,.2f}"

    def refund(self):
        if self.status != "SUCCESS":
            return "Cannot refund — payment not successful."
        self.__wallet_balance += self.amount
        self.status = "REFUNDED"
        return f"♻️  Refund of {self.currency} {self.amount:,.2f} added back to wallet."


# ─── Payment Gateway (Orchestrator) ──────────────────────────────────────────
class PaymentGateway:
    def __init__(self):
        self.__transactions = []

    def pay(self, payment: Payment):
        """Polymorphism — same interface for all payment types."""
        result = payment.process()
        self.__transactions.append(payment)
        print(result)
        print(payment.get_receipt())
        return payment

    def get_history(self):
        print("\n─── Transaction History ──────────────")
        for txn in self.__transactions:
            print(f"  {txn}")
        print("──────────────────────────────────────")

    def total_collected(self):
        return sum(p.amount for p in self.__transactions if p.status == "SUCCESS")


# ─── Usage ────────────────────────────────────────────────────────────────────
gateway = PaymentGateway()

# Different payment methods — same process() interface (Polymorphism)
card = CreditCardPayment(5000, "1234567890123456", "Alice")
upi = UPIPayment(1500, "alice@upi")
wallet = WalletPayment(800, 2000)

gateway.pay(card)
gateway.pay(upi)
gateway.pay(wallet)

# Refund one
print(upi.refund())

gateway.get_history()
print(f"\nTotal collected: INR {gateway.total_collected():,.2f}")

Processing card ****-****-****-3456...
✅ Card payment of INR 5,000.00 successful!
─── Receipt ───────────────────
  TXN ID   : BB1F82E4
  Amount   : INR 5,000.00
  Status   : SUCCESS
  Time     : 2026-03-27 14:14:31
  Type     : CreditCardPayment
───────────────────────────────
Sending payment request to alice@upi...
✅ UPI payment of INR 1,500.00 to alice@upi successful!
─── Receipt ───────────────────
  TXN ID   : A1B3160C
  Amount   : INR 1,500.00
  Status   : SUCCESS
  Time     : 2026-03-27 14:14:31
  Type     : UPIPayment
───────────────────────────────
✅ Wallet payment of INR 800.00 successful! Remaining: 1,200.00
─── Receipt ───────────────────
  TXN ID   : B8529763
  Amount   : INR 800.00
  Status   : SUCCESS
  Time     : 2026-03-27 14:14:31
  Type     : WalletPayment
───────────────────────────────
♻️  Refund of INR 1,500.00 to alice@upi initiated.

─── Transaction History ──────────────
  CreditCardPayment(****-****-****-3456, INR 5,000.00)
  UPIPayment(id=A1B3160C, amount=150